# Track A Historia: notebook de revision y experimentos para David

Este notebook esta pensado para dos cosas:

1. **Revisar con David que estamos entrenando lo que decimos entrenar.**  
   DMLPA aqui es un `BaseFeaturesExtractor` tipo Transformer sobre una ventana `VecFrameStack(30)`. RecurrentPPO es el comparador con memoria LSTM real.

2. **Probar ideas de convergencia sin romper el protocolo.**  
   El notebook imprime y ejecuta comandos reproducibles sobre el entorno final, compara contra la mejor estatica del mismo panel, y rankea por metricas de resiliencia externas, no por `reward_total`.

La pregunta cientifica de este carril es:

> Con las mismas variables de decision de Garrido, el uso de historia reciente mejora la resiliencia bajo estres no saturado frente a la mejor politica estatica?


## Que cambio frente a los notebooks originales de David

Los notebooks originales son utiles para entender la intuicion de DMLPA, pero no deben ser la fuente final de entrenamiento porque habia drift de protocolo:

- Algunas versiones no fijaban `risk_occurrence_mode=thesis_periodic`.
- Algunas no fijaban `raw_material_flow_mode=kit_equivalent_order_up_to`.
- Una version entrenaba contra el entorno crudo (`_se`) en vez del wrapper de entrenamiento.
- El nombre DMLPA no siempre garantizaba que PPO recibiera `policy_kwargs` con el extractor.

En este notebook, DMLPA se porta al repo y se ejecuta desde `scripts/run_thesis_decision_ppo_smoke.py`, con el mismo runner que evalua RecurrentPPO, MLP, grilla estatica y metricas.


## Contrato experimental fijo

**Arquitecturas**

- `recurrent_ppo`: `MlpLstmPolicy`, memoria LSTM oculta.
- `dmlpa_ppo`: PPO normal + extractor DMLPA/Transformer sobre `VecFrameStack(30)`. Ve historia reciente, pero no mantiene estado oculto recurrente.

**Espacios de decision**

- `thesis_factorized`: contrato discreto de Garrido, `[I_{t,S}, S]`.
- `continuous_it_s`: mismas dos variables conceptuales de Garrido, pero buffer continuo.

**Riesgos**

- `severe_training`: maximo estres antes de guerra.
- `war_stress_v1`: extension explicita tipo guerra.

**Entorno final**

- `risk_occurrence_mode=thesis_periodic`
- `raw_material_flow_mode=kit_equivalent_order_up_to`
- `raw_material_order_up_to_multiplier=2.0`
- `stochastic_pt=True`, `stochastic_pt_spread=1.0`
- `observation_version=v5`
- `observation_mode=env_sdm_history_reward`
- `reward_profile=ret_ladder_steep`


## Criterio de victoria

No usamos `reward_total` como victoria. El reward es una herramienta de aprendizaje; la victoria se decide contra la mejor estatica del mismo panel y riesgo.

Metricas primarias:

- `ret_mean_all_orders_zero_unfulfilled`
- `flow_fill_rate`
- `stockout_week_pct` (menor es mejor)
- `ret_p10_all`

Regla de parada:

- Si las 8 configs pierden frente a la mejor estatica por mas de `0.01` en fill y `0.02` en ReT/flow-fill, parar Track A.
- Si alguna config empata fill dentro de `0.01` y mejora ReT all-orders o flow-fill, pasa a confirmatorio.


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
from typing import Iterable

SEED = 4242
REPO_URL = "https://github.com/Thom-320/scres-ia.git"
TARGET_REF = os.environ.get("SCRESIA_TARGET_REF", "codex/garrido-postfix-reruns")

IS_KAGGLE = Path("/kaggle/working").exists()
CURRENT_DIR = Path.cwd()
DEFAULT_REPO_DIR = CURRENT_DIR if (CURRENT_DIR / "scripts" / "run_track_a_exhaustion_sweep.py").exists() else Path("/kaggle/working/scres-ia")
REPO_DIR = Path(os.environ.get("SCRESIA_REPO_DIR", str(DEFAULT_REPO_DIR)))
OUTPUT_ROOT = Path(os.environ.get("SCRESIA_OUTPUT_ROOT", "/kaggle/working/track_a_history_outputs" if IS_KAGGLE else "outputs/notebook_david_track_a_history"))

# Safe defaults: review first, then intentionally turn runs on.
RUN_SMOKES = False
RUN_SCREENING = False
RUN_CONFIRMATORY = False
REQUIRE_COMPATIBLE_GPU_FOR_SCREENING = True

print(json.dumps({
    "is_kaggle": IS_KAGGLE,
    "target_ref": TARGET_REF,
    "repo_dir": str(REPO_DIR),
    "output_root": str(OUTPUT_ROOT),
    "run_smokes": RUN_SMOKES,
    "run_screening": RUN_SCREENING,
    "run_confirmatory": RUN_CONFIRMATORY,
}, indent=2))


In [ ]:
def run(cmd: list[str], cwd: Path | None = None, check: bool = True) -> subprocess.CompletedProcess:
    print("$", " ".join(map(str, cmd)))
    return subprocess.run(cmd, cwd=str(cwd) if cwd else None, check=check)

if not (REPO_DIR / "scripts" / "run_track_a_exhaustion_sweep.py").exists():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    run(["git", "clone", "--depth", "1", "--branch", TARGET_REF, REPO_URL, str(REPO_DIR)])
else:
    print("Using repo:", REPO_DIR)

run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], cwd=REPO_DIR)
import pandas as pd
run(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR)


In [ ]:
def compatible_torch_device() -> tuple[str, dict[str, object]]:
    import torch

    info: dict[str, object] = {"cuda_available": bool(torch.cuda.is_available())}
    if not torch.cuda.is_available():
        info["device"] = "cpu"
        info["reason"] = "CUDA not available"
        return "cpu", info

    capability = torch.cuda.get_device_capability(0)
    name = torch.cuda.get_device_name(0)
    info.update({"device_name": name, "capability": capability})
    if capability[0] < 7:
        info["device"] = "cpu"
        info["reason"] = "P100/sm_60 is incompatible with current Kaggle PyTorch; request T4/A100"
        return "cpu", info

    info["device"] = "cuda"
    info["reason"] = "compatible GPU"
    return "cuda", info

DEVICE_ARG, DEVICE_INFO = compatible_torch_device()
print(json.dumps(DEVICE_INFO, indent=2, default=str))


## Inspeccion del DMLPA real que se esta entrenando

Esta celda imprime el extractor desde el repo. Es importante que David pueda verificar que no estamos entrenando un MLP normal con una etiqueta bonita.


In [ ]:
import inspect
import importlib.util

smoke_path = REPO_DIR / "scripts" / "run_thesis_decision_ppo_smoke.py"
spec = importlib.util.spec_from_file_location("track_a_smoke", smoke_path)
track_a_smoke = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(track_a_smoke)

print(inspect.getsource(track_a_smoke.DMLPAPositionalExtractor))


## Mapa de metricas

La confusion principal fue mezclar dos superficies:

- **Reward step-level**: proxy denso para entrenar PPO.
- **Metricas paper-facing**: evaluacion order-level y cola de resiliencia.

Para discutir con David: si una arquitectura sube `reward_total` pero no sube `ret_mean_all_orders_zero_unfulfilled`, `flow_fill_rate`, `ret_p10_all` o reduce `stockout_week_pct`, no es una victoria cientifica.


In [ ]:
PRIMARY_METRICS = [
    "ret_mean_all_orders_zero_unfulfilled",
    "flow_fill_rate",
    "stockout_week_pct",
    "ret_p10_all",
]

SUMMARY_RANK_COLUMNS = [
    "algo",
    "action_space_mode",
    "risk_level",
    "status",
    "delta_ret_all_orders",
    "delta_flow_fill",
    "delta_fill",
    "delta_ret_p10_all",
    "delta_stockout_week_pct",
    "best_static_policy",
    "ppo_ret_all_orders",
    "best_static_ret_all_orders",
    "ppo_flow_fill",
    "best_static_flow_fill",
    "ppo_fill",
    "best_static_fill",
]

pd.DataFrame({
    "metric": PRIMARY_METRICS,
    "interpretation": [
        "Mean ReT including unfulfilled orders as zero",
        "Quantity/flow service level, catches hidden shortages",
        "Percent of decision weeks with stockout/backlog; lower is better",
        "Lower-tail ReT; asks whether bad cases improve",
    ],
})


## Smokes cortos

Los smokes prueban construccion de arquitectura, wrapper, metricas y `policy_summary.csv`. No prueban performance.

Para correrlos, cambia `RUN_SMOKES = True` en la celda de configuracion.


In [ ]:
def smoke_command(label: str, algo: str, action_space: str, risk: str, seed: int) -> list[str]:
    return [
        sys.executable,
        "scripts/run_thesis_decision_ppo_smoke.py",
        "--label", label,
        "--output-root", str(OUTPUT_ROOT / "smokes"),
        "--train-timesteps", "1024",
        "--eval-episodes", "1",
        "--seed", str(seed),
        "--algo", algo,
        "--device", DEVICE_ARG,
        "--history-window", "30",
        "--risk-level", risk,
        "--risk-occurrence-mode", "thesis_periodic",
        "--raw-material-flow-mode", "kit_equivalent_order_up_to",
        "--raw-material-order-up-to-multiplier", "2.0",
        "--action-space-mode", action_space,
        "--inventory-period-mode", "thesis_strict",
        "--max-steps", "8",
        "--include-static-grid",
        "--no-eval-ai-on-garrido-cfis",
        "--n-steps", "64",
        "--batch-size", "32",
        "--n-epochs", "1",
        "--reward-mode", "ReT_ladder_v1",
        "--ret-ladder-w-sc", "0.80",
        "--ret-ladder-w-rc", "0.20",
        "--ret-ladder-w-ef", "0.00",
        "--ret-ladder-gate-beta", "20.0",
        "--stochastic-pt",
        "--stochastic-pt-spread", "1.0",
        "--train-cfis", "31",
        "--garrido-cfis", "31",
        "--train-risk-profile", risk,
        "--eval-risk-profile", risk,
    ]

SMOKE_COMMANDS = [
    smoke_command("david_smoke_dmlpa_thesis_severe", "dmlpa_ppo", "thesis_factorized", "severe_training", 101),
    smoke_command("david_smoke_recurrent_continuous_war", "recurrent_ppo", "continuous_it_s", "war_stress_v1", 102),
]

for cmd in SMOKE_COMMANDS:
    print(" ".join(cmd), "\n")

if RUN_SMOKES:
    for cmd in SMOKE_COMMANDS:
        run(cmd, cwd=REPO_DIR)


## Screening serio de 8 configs

Este es el experimento que agota Track A Historia:

`2 arquitecturas x 2 espacios x 2 riesgos = 8 runs`

Presupuesto screening: `100k` timesteps, panel `Cf31-90`, `max_steps=260`, 1 seed.  
En Kaggle, usar T4/A100. Si Kaggle asigna P100, este notebook debe advertirlo antes de gastar tiempo.


In [ ]:
def screening_command(
    *,
    label_prefix: str = "david_track_a_history_screen",
    train_timesteps: int = 100_000,
    panel_cfis: str = "31-90",
    eval_episodes: int = 5,
    max_steps: int = 260,
    history_window: int = 30,
    algos: Iterable[str] = ("recurrent_ppo", "dmlpa_ppo"),
    action_spaces: Iterable[str] = ("thesis_factorized", "continuous_it_s"),
    risks: Iterable[str] = ("severe_training", "war_stress_v1"),
) -> list[str]:
    return [
        sys.executable,
        "scripts/run_track_a_exhaustion_sweep.py",
        "--label-prefix", label_prefix,
        "--output-root", str(OUTPUT_ROOT / "screening"),
        "--algos", *list(algos),
        "--action-space-modes", *list(action_spaces),
        "--reward-profiles", "ret_ladder_steep",
        "--risk-levels", *list(risks),
        "--pt-profiles", "stoch_pt_hist",
        "--use-cf-risk-profile",
        "--panel-cfis", panel_cfis,
        "--train-timesteps", str(train_timesteps),
        "--eval-episodes", str(eval_episodes),
        "--max-steps", str(max_steps),
        "--n-steps", "1024",
        "--batch-size", "64",
        "--n-epochs", "10",
        "--history-window", str(history_window),
        "--device", DEVICE_ARG,
        "--stop-on-error",
    ]

SCREENING_CMD = screening_command()
print(" ".join(SCREENING_CMD))

if RUN_SCREENING:
    if REQUIRE_COMPATIBLE_GPU_FOR_SCREENING and DEVICE_ARG != "cuda":
        raise RuntimeError(f"Serious screening requires compatible GPU. Current device info: {DEVICE_INFO}")
    run(SCREENING_CMD, cwd=REPO_DIR)


## Cargar resultados de screening

Esta celda busca `sweep_summary.csv` en el output local/Kaggle. Tambien puedes subir un CSV manualmente y poner su ruta en `MANUAL_SUMMARY_PATH`.


In [ ]:
MANUAL_SUMMARY_PATH = ""  # optional, e.g. "/kaggle/input/run-output/sweep_summary.csv"

def find_summary_files() -> list[Path]:
    candidates: list[Path] = []
    if MANUAL_SUMMARY_PATH:
        candidates.append(Path(MANUAL_SUMMARY_PATH))
    for root in [OUTPUT_ROOT, Path("/kaggle/working"), Path("/kaggle/input"), Path.cwd()]:
        if root.exists():
            candidates.extend(root.glob("**/sweep_summary.csv"))
    return sorted(set(candidates), key=lambda p: p.stat().st_mtime if p.exists() else 0, reverse=True)

summary_files = find_summary_files()
print("Found", len(summary_files), "summary files")
for i, p in enumerate(summary_files[:10]):
    print(i, p)

summary = pd.read_csv(summary_files[0]) if summary_files else pd.DataFrame()
summary.head() if not summary.empty else "No summary loaded yet"


In [ ]:
def rank_summary(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    work = df.copy()
    for col in ["delta_ret_all_orders", "delta_flow_fill", "delta_fill", "delta_ret_p10_all", "delta_stockout_week_pct"]:
        if col not in work.columns:
            work[col] = 0.0
    work["promote"] = (
        ((work["delta_fill"] >= -0.01) & ((work["delta_ret_all_orders"] > 0.0) | (work["delta_flow_fill"] > 0.0)))
        | (work["delta_fill"] >= 0.01)
        | (work["delta_ret_all_orders"] >= 0.02)
        | (work["delta_flow_fill"] >= 0.02)
    )
    work["stop_track_a_loss"] = (
        (work["delta_fill"] < -0.01)
        & (work["delta_ret_all_orders"] < -0.02)
        & (work["delta_flow_fill"] < -0.02)
    )
    return work.sort_values(
        by=["delta_ret_all_orders", "delta_flow_fill", "delta_fill", "delta_ret_p10_all"],
        ascending=[False, False, False, False],
    )

ranked = rank_summary(summary)
if ranked.empty:
    print("No summary loaded yet.")
else:
    cols = [c for c in SUMMARY_RANK_COLUMNS + ["promote", "stop_track_a_loss"] if c in ranked.columns]
    display(ranked[cols].head(12))
    print("all_runs_fail_stop_rule:", bool(ranked["stop_track_a_loss"].all()))
    print("promoted_count:", int(ranked["promote"].sum()))


## Confirmatorio top-2

Solo correr si el screening produce configs promovibles. Presupuesto: `500k x 3 seeds` para top-2.


In [ ]:
def confirmatory_commands(df: pd.DataFrame, top_n: int = 2) -> list[list[str]]:
    if df.empty or "promote" not in df.columns:
        return []
    promoted = df[df["promote"]].head(top_n)
    commands: list[list[str]] = []
    for _, row in promoted.iterrows():
        for seed in [4242, 4243, 4244]:
            commands.append(screening_command(
                label_prefix=f"confirm_{row['algo']}_{row['action_space_mode']}_{row['risk_level']}_seed{seed}",
                train_timesteps=500_000,
                panel_cfis="31-90",
                eval_episodes=10,
                max_steps=260,
                history_window=30,
                algos=(str(row["algo"]),),
                action_spaces=(str(row["action_space_mode"]),),
                risks=(str(row["risk_level"]),),
            ) + ["--seed", str(seed)])
    return commands

CONFIRMATORY_COMMANDS = confirmatory_commands(ranked)
for cmd in CONFIRMATORY_COMMANDS:
    print(" ".join(cmd), "\n")

if RUN_CONFIRMATORY:
    if REQUIRE_COMPATIBLE_GPU_FOR_SCREENING and DEVICE_ARG != "cuda":
        raise RuntimeError(f"Confirmatory run requires compatible GPU. Current device info: {DEVICE_INFO}")
    for cmd in CONFIRMATORY_COMMANDS:
        run(cmd, cwd=REPO_DIR)


## Sandbox de ideas para David

Regla: cambiar una cosa a la vez y comparar siempre contra la misma estatica.

Ideas razonables para convergencia:

1. **Historia**: `history_window = 10, 30, 60`.  
   Si 60 ayuda a DMLPA pero no a RecurrentPPO, la ventana explicita importa.

2. **DMLPA architecture**: `features_dim = 120/180`, `num_layers = 2/4`, `heads = 6/12`.  
   Primero ablations chicas; no agrandar antes de ver señal.

3. **DMLPA faithful vs positional**: correr una variante sin positional encoding para separar “arquitectura de David” de “mejora posicional”.

4. **Reward sin reward-hacking**: probar rewards mas empinados solo si mejoran `ret_p10_all`, `flow_fill_rate`, `stockout_week_pct`, no solo `reward_total`.

5. **Curriculum de riesgo**: entrenar primero en `severe_training`, evaluar en `war_stress_v1`; o entrenar mezclado. Esto puede ayudar si guerra desde cero es demasiado ruidoso.

6. **Continuous It,S**: mantener las mismas variables de Garrido pero evitar la grilla gruesa del DOE. Si gana aqui, el claim es mas limpio que Track B.


In [ ]:
# Build a custom experiment command. Edit this cell with David.
CUSTOM = {
    "label_prefix": "david_custom_probe",
    "algos": ("dmlpa_ppo",),
    "action_spaces": ("continuous_it_s",),
    "risks": ("war_stress_v1",),
    "history_window": 30,
    "train_timesteps": 30_000,
    "panel_cfis": "31,41,51,61,71,81",
    "eval_episodes": 2,
    "max_steps": 120,
}

CUSTOM_CMD = screening_command(**CUSTOM)
print(" ".join(CUSTOM_CMD))


## Checklist para una propuesta de David

Antes de lanzar GPU seria, cada idea deberia responder:

- Que cambia exactamente?
- Por que deberia ayudar a aprender historia o resiliencia?
- Contra que baseline estatica se compara?
- Que metrica primaria espera mejorar?
- Cual es la stop rule?
- Cuanto cuesta en GPU?

Si la propuesta no puede contestar eso, todavia es una intuicion, no un experimento.


In [ ]:
# Pack outputs for download from Kaggle/local.
archive_base = Path("/kaggle/working/track_a_history_outputs") if IS_KAGGLE else Path("outputs/notebook_david_track_a_history")
if OUTPUT_ROOT.exists():
    archive = shutil.make_archive(str(archive_base), "zip", root_dir=str(OUTPUT_ROOT))
    print("archive:", archive)
else:
    print("No output root found yet:", OUTPUT_ROOT)
